In [3]:
import torch
import transformers
import datasets
import peft
import trl
import accelerate
import wandb

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA:", torch.cuda.is_available())
print("wandb:", wandb.__version__)

Torch: 2.5.1+cu121
Transformers: 5.9.0
Datasets: 4.8.5
PEFT: 0.19.1
TRL: 1.5.1
CUDA: True
wandb: 0.27.0


#Bringing all the required libraries

In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()
login(token=os.getenv("HF_TOKEN"))

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

modelName = "google/gemma-2-2b-it"

bnbConfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # safer than bfloat16 on many consumer GPUs
)

tokenizer = AutoTokenizer.from_pretrained(modelName)

model = AutoModelForCausalLM.from_pretrained(
    modelName,
    device_map="auto",
    quantization_config=bnbConfig
)

Loading weights: 100%|██████████| 288/288 [00:01<00:00, 195.92it/s]


In [6]:
input_text = "How are you today? how have you been recently?"
input_ids = tokenizer(input_text, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=200)
print(outputs)
print(tokenizer.decode(outputs[0]))

tensor([[     2,   2299,    708,    692,   3646, 235336,   1368,    791,    692,
           1125,   7378, 235336,    109, 235285, 235303, 235262,   3900,   1578,
         235269,   7593,    692, 235341,    590, 235303,    524,   1125,  13572,
            675,   1160,    578,   1009,   3749,   7340, 235269,    901,   8691,
         235269,    590, 235303, 235262,   4915, 235265,   2250,   1105,    692,
         235336,  44416, 235248,    108,    107]], device='cuda:0')
<bos>How are you today? how have you been recently?

I'm doing well, thank you! I've been busy with work and some personal projects, but overall, I'm happy. How about you? 😊 
<end_of_turn>


# Fine-tuning Steps for Gemma 2 Using LoRA On top of Qlora 4 bit Quantizattion

In [7]:
import os
#!pip uninstall -y datasets pyarrow
#!pip install pyarrow datasets

from transformers import (
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)

In [8]:
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)


In [9]:
from datasets import load_dataset

In [10]:
from trl import SFTTrainer

In [ ]:
# in the terminal run : wandb login

#then put below API key

#wandb_api_key = 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

In [11]:
run = wandb.init(
    project='Fine-tune Gemma-2-2b-it on doctor-healthcare dataset',
    job_type="training",
    anonymous="allow"
)

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\stron\_netrc.
wandb: Currently logged in as: saba-rezaee-khavas (saba-rezaee-khavas-umass-lowell) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


#Loading Model and Tokeniuzers

In [12]:
torch_dtype = torch.float16
attn_implementation = "eager"

QLORA config

In [13]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)


Load Model

In [14]:
base_model = "google/gemma-2-2b-it"
dataset_name = df
new_model = "Gemma-2-2b-it"

In [15]:
# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

Loading weights: 100%|██████████| 288/288 [00:01<00:00, 192.43it/s]


In [16]:
model

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)


In [ ]:
print(model.named_modules())

The function traverses the quantized model, identifies all Linear4bit layers ( all the layer that has been quantized when the model was loaded load_in_4bit=True) created by BitsAndBytes, extracts their module names, and returns them as LoRA target modules. This allows PEFT to automatically inject LoRA adapters into the model's attention and feed-forward projection layers without manually specifying each layer


some people actually hard code that as modules = ['down_proj', 'o_proj', 'up_proj', 'v_proj', 'gate_proj', 'q_proj', 'k_proj']


In [17]:
import bitsandbytes as bnb

def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

modules = find_all_linear_names(model)

In [18]:
print(modules)

['v_proj', 'up_proj', 'o_proj', 'down_proj', 'gate_proj', 'q_proj', 'k_proj']


In [19]:
import pandas as pd
df = pd.read_csv('doctor-healthcare-100k/Doctor-HealthCare-100k.csv')
df.head()

,instruction,input,output
0,"If you are a doctor, please answer the medical...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"If you are a doctor, please answer the medical...",My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"If you are a doctor, please answer the medical...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"If you are a doctor, please answer the medical...",lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,"If you are a doctor, please answer the medical...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [20]:
def format_chatml(example):
    text = (
        "<|im_start|>system\n"
        f"{example['instruction']}"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['input']}"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['output']}"
        "<|im_end|>"
    )

    return {"text": text}

In [21]:
from datasets import Dataset # Import the Dataset class

# Convert pandas DataFrame to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Now map the format_example function to the Hugging Face Dataset
dataset = hf_dataset.map(format_chatml)

Map: 100%|██████████| 112156/112156 [00:03<00:00, 35204.55 examples/s]


In [22]:
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 112156
})

In [23]:
dataset['text'][1]


"<|im_start|>system\nIf you are a doctor, please answer the medical questions based on the patient's description.<|im_end|>\n<|im_start|>user\nMy baby has been pooing 5-6 times a day for a week. In the last few days it has increased to 7 and they are very watery with green stringy bits in them. He does not seem unwell i.e no temperature and still eating. He now has a very bad nappy rash from the pooing ...help!<|im_end|>\n<|im_start|>assistant\nHi... Thank you for consulting in Chat Doctor. It seems your kid is having viral diarrhea. Once it starts it will take 5-7 days to completely get better. Unless the kids having low urine output or very dull or excessively sleepy or blood in motion or green bilious vomiting...you need not worry. There is no need to use antibiotics unless there is blood in the motion. Antibiotics might worsen if unnecessarily used causing antibiotic associated diarrhea. I suggest you use zinc supplements (Z&D Chat Doctor.<|im_end|>"

<|im_start|>system
If you are a doctor, please answer the medical questions based on the patient's description.<|im_end|>
<|im_start|>user
Hi i am a teenager. about 2 days a go i found about 5 slim lumps across my forehead. do you no what this could be my mum says that it is just boils but im worried could help me. also i have been have a lot of headaches/migraines as well. it also herts when i touch them.<|im_end|>
<|im_start|>assistant
Hi, Dear I studied your query in all it details and I understood your concerns. Cause - On whatever limited facts given you seem to have Acne, or pimples, and they are painful to touch. The migraine or headaches is a separate ailment and don't correlate with painful acne on forehead. So don't worry.Hence, To reduce your worry Please consult for opinion from ER doctor. Plz hit thanks and write excellent Reviews if this would resolve your query. Plz don't worry and do Welcome for any further query in this regard to me. Have a Good Day. Chat Doctor. N.<|im_end|>

In [26]:
# LoRA config LoRA adds small, low-rank matrices to each layer, allowing only these matrices to be trained.
#This minimizes the computational load and memory needed.
tokenizer.chat_template = None  # this disables any built-in chat formatting
peft_config = LoraConfig(
    r=16,     #  the rank in LoRA directly affects the number of trainable parameters in the model. more rank more parameters to train
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",  # dont train biases, only train lora adapters
    task_type="CAUSAL_LM",
    target_modules=modules,
)
model = get_peft_model(model, peft_config)

c:\Users\stron\anaconda3\envs\llm_clean\lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\stron\anaconda3\envs\llm_clean\lib\site-packages\peft\tuners\tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [27]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): Gemma2ForCausalLM(
          (model): Gemma2Model(
            (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
            (layers): ModuleList(
              (0-25): 26 x Gemma2DecoderLayer(
                (self_attn): Gemma2Attention(
                  (q_proj): lora.Linear4bit(
                    (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=2304, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=2048, bias=False)
                    )
                    (lora_em

In [28]:
dataset = dataset.train_test_split(test_size=0.1)
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 100940
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 11216
    })
})

#Training

In [29]:
training_arguments = TrainingArguments(
    output_dir=new_model,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    report_to="wandb"
)

# Setting sft parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_arguments,
)

model.config.use_cache = False
trainer.train()

Tokenizing eval dataset: 100%|██████████| 11216/11216 [00:04<00:00, 2768.30 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10094,2.238945,1.936818,1.956838,5759367.000000,0.584328
20188,2.287543,1.875842,1.887090,11551050.000000,0.594285
30282,1.840909,1.832501,1.822124,17332766.000000,0.601112
40376,1.327758,1.803501,1.812311,23135086.000000,0.606039
50470,1.724057,1.796705,1.795332,28928259.000000,0.607117


TrainOutput(global_step=50470, training_loss=1.8965417022820343, metrics={'train_runtime': 126017.9821, 'train_samples_per_second': 0.801, 'train_steps_per_second': 0.4, 'total_flos': 3.549991372136248e+17, 'train_loss': 1.8965417022820343, 'epoch': 1.0})